# Gemini 장면 분석 — 객체 + 대사 + 프레임 이미지

**목표:** YOLO 객체 + Whisper 대사 + 씬 분할 영상 프레임(3장)을 Gemini에 전달하여 상황/감정/욕구 분석

**입력:**
- `whisper_metadata.csv` (Whisper 타임스탬프 + 대사)
- `Yolo_scene_objects.csv` (YOLO 객체 탐지 결과)
- → 두 파일을 타임스탬프 기준으로 병합 → `merged_whisper_yolo.csv`
- `hyuck1_context_scenes/` (Whisper 문맥 기준 씬 분할 영상 클립)

**출력:**
- 씬별 상황/감정/욕구 텍스트 → CSV 저장 + DB 직접 업로드 (analysis_scene_final)

## 0. 환경 설정

In [ ]:
# 패키지 설치
!pip install -q google-generativeai opencv-python psycopg2-binary pillow tqdm
print("패키지 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

BASE      = "/content/drive/MyDrive/SampleVideo.Test"
SCENE_DIR = os.path.join(BASE, "hyuck1_context_scenes")

# merged_whisper_yolo.csv 직접 로드
merged_csv = os.path.join(BASE, 'merged_whisper_yolo.csv')
if os.path.exists(merged_csv):
    df = pd.read_csv(merged_csv)
    print(f"merged_whisper_yolo.csv 로드: {len(df)}개 씬")
else:
    raise FileNotFoundError("merged_whisper_yolo.csv 없음 — merge_whisper_yolo_colab.ipynb 먼저 실행하세요.")

# 씬 폴더 확인
print(f"[씬 폴더] ", end="")
if os.path.exists(SCENE_DIR):
    clips = sorted([f for f in os.listdir(SCENE_DIR) if f.endswith('.mp4')])
    print(f"{len(clips)}개 클립 OK")
else:
    print("MISSING — hyuck1_context_scenes 폴더를 확인하세요")

print(f"\n총 {len(df)}개 씬")
print(df[['context_num','filename','start_sec','end_sec']].head(3))


In [ ]:
# ⛔ 이 셀은 사용하지 않습니다 — cell-6에서 Gemini 모델을 초기화합니다.
# (구 버전 gemini-1.5-flash 코드 — 실행 금지)
print("cell-5 skip — cell-6을 실행하세요.")

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# API 키 로드
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    print("Colab Secrets에서 API 키 로드 완료!")
except:
    API_KEY = "여기에_API_키_입력"
    print("직접 입력된 API 키 사용")

genai.configure(api_key=API_KEY)

# 모델 초기화 — 2.5-flash 우선, 없으면 1.5-flash 자동 fallback
MODEL_CANDIDATES = [
    'gemini-2.5-flash-preview-04-17',
    'gemini-2.5-flash',
    'gemini-2.5-flash-preview',
    'gemini-1.5-flash',
]

model = None
for model_name in MODEL_CANDIDATES:
    try:
        m = genai.GenerativeModel(model_name)
        # 간단한 테스트 호출
        m.generate_content("test")
        model = m
        print(f"✅ 모델 로드 성공: {model_name}")
        break
    except Exception as e:
        print(f"⚠️ {model_name} 실패: {str(e)[:60]}")

if model is None:
    raise RuntimeError("사용 가능한 Gemini 모델이 없습니다. API 키를 확인하세요.")

## 3. 프롬프트 템플릿 & 분석 함수

In [ ]:
import cv2
import time
from PIL import Image
import google.generativeai as genai

# ── 1. 프레임 추출 함수 ──────────────────────────────────────────
def extract_frames_from_clip(clip_path):
    """씬 클립에서 시작(10%) / 중간(50%) / 끝(90%) 3장 추출 → PIL Image 리스트"""
    cap = cv2.VideoCapture(clip_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []

    targets = [
        max(1,           int(total * 0.1)),
        int(total * 0.5),
        min(total - 1,   int(total * 0.9)),
    ]

    frames = []
    for pos in targets:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
    cap.release()
    return frames


# ── 2. 프롬프트 템플릿 ───────────────────────────────────────────
_PROMPT_TEMPLATE = (
    "아래는 한국 TV 드라마의 한 씬에 대한 정보입니다.\n\n"
    "{context_block}\n\n"
    "[화면 속 탐지된 객체]\n"
    "{detected_objects}\n\n"
    "[대사]\n"
    "{dialogue_text}\n\n"
    "위 객체와 대사 정보를 반드시 참고하여, 이 씬을 분석하고 아래 세 항목을 각각 정확히 1문장으로 작성하세요.\n"
    "반드시 'label: 내용' 형식을 지켜야 합니다.\n\n"
    "상황: 이 씬에 등장하는 장면, 배경, 인물의 행동을 묘사하세요. (감정·느낌 표현 금지)\n"
    "감정: 이 씬이 시청자에게 전달하는 감성적 분위기나 정서를 표현하세요. (상황 묘사 금지)\n"
    "욕구: 이 씬이 타겟하는 시청자의 내면적 니즈나 욕구를 서술하세요. (감정 단어·상황 묘사 금지)\n\n"
    "예시:\n"
    "상황: 퇴근 후 집에 돌아온 직장인이 소파에 앉아 따뜻한 음료를 마시는 장면이다.\n"
    "감정: 하루의 피로가 녹아드는 포근하고 안도감 있는 분위기를 전달한다.\n"
    "욕구: 바쁜 일상 속에서 잠깐의 휴식과 자신을 위한 작은 여유를 원하는 사람에게 어울린다.\n\n"
    "반드시 한국어로만 작성하고, 위 예시처럼 세 줄로만 답하세요."
)


# ── 3. 씬 분석 함수 ─────────────────────────────────────────────
def analyze_scene(row, scene_dir, model):
    """씬 1개 분석: 클립 프레임 3장 + 객체 + 대사 → Gemini"""
    clip_path = os.path.join(scene_dir, str(row['filename']))

    if os.path.exists(clip_path):
        frames = extract_frames_from_clip(clip_path)
    else:
        frames = []
        print(f"  ⚠️ 클립 없음: {row['filename']}")

    dialogue = row['dialogue'] if pd.notna(row['dialogue']) else '(대사 없음)'
    objects  = row['detected_objects'] if pd.notna(row['detected_objects']) else '(탐지 없음)'

    prompt = _PROMPT_TEMPLATE.format(
        context_block=f"씬 구간: {row['start_sec']:.1f}s ~ {row['end_sec']:.1f}s",
        detected_objects=objects,
        dialogue_text=dialogue,
    )

    content = frames + [prompt]
    try:
        response = model.generate_content(content)
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"


# ── 4. 파싱 함수 ────────────────────────────────────────────────
def parse_result(result_text):
    """Gemini 응답 → 상황/감정/욕구 dict"""
    parsed = {'상황': '', '감정': '', '욕구': ''}
    for line in result_text.split('\n'):
        line = line.strip()
        for label in ['상황', '감정', '욕구']:
            if line.startswith(f'{label}:') or line.startswith(f'{label} :'):
                parsed[label] = line.split(':', 1)[1].strip()
    return parsed


print("✅ 프레임 추출 / 프롬프트 / 분석 함수 정의 완료!")


## 4. 전체 씬 분석 실행

In [ ]:
from tqdm.auto import tqdm

results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Gemini 분석"):
    # Gemini API 호출 (씬 분할 클립 기준)
    raw_result = analyze_scene(row, SCENE_DIR, model)
    parsed = parse_result(raw_result)
    
    results.append({
        'context_num': row['context_num'],
        'filename': row['filename'],
        'start_sec': row['start_sec'],
        'end_sec': row['end_sec'],
        'detected_objects': row['detected_objects'],
        'dialogue': row['dialogue'],
        'situation': parsed['상황'],
        'emotion': parsed['감정'],
        'desire': parsed['욕구'],
        'raw_response': raw_result,
    })
    
    # API Rate Limit 방지 (분당 60회 제한)
    time.sleep(1.5)
    
    # 10개마다 진행 상황 출력
    if (idx + 1) % 10 == 0:
        print(f"  [{idx+1}/{len(df)}] 최근 결과:")
        print(f"    상황: {parsed['상황'][:50]}...")
        print(f"    감정: {parsed['감정'][:50]}...")
        print(f"    욕구: {parsed['욕구'][:50]}...")

df_results = pd.DataFrame(results)
print(f"\n분석 완료! {len(df_results)}개 씬")

## 5. 결과 확인

In [ ]:
from IPython.display import display, HTML

# 상위 10개 결과 미리보기
print("=" * 60)
print("  Gemini 장면 분석 결과 미리보기")
print("=" * 60)

for _, row in df_results.head(10).iterrows():
    print(f"\n--- 씬 {row['context_num']} ({row['start_sec']:.1f}s ~ {row['end_sec']:.1f}s) ---")
    print(f"  객체: {row['detected_objects']}")
    d = row['dialogue'] if pd.notna(row['dialogue']) else '(대사 없음)'
    print(f"  대사: {d[:60]}...")
    print(f"  상황: {row['situation']}")
    print(f"  감정: {row['emotion']}")
    print(f"  욕구: {row['desire']}")

# 파싱 실패 건수 확인
empty_count = df_results[df_results['situation'] == ''].shape[0]
error_count = df_results[df_results['raw_response'].str.startswith('ERROR')].shape[0]
print(f"\n파싱 실패: {empty_count}건")
print(f"API 오류: {error_count}건")
print(f"성공: {len(df_results) - empty_count - error_count}건")

## 6. CSV 저장 & Drive 백업

In [ ]:
import shutil

# CSV 저장
output_csv = 'hyuck1_gemini_analysis.csv'
df_results.to_csv(output_csv, index=False, encoding='utf-8-sig')

# Drive 백업
drive_path = os.path.join(BASE, output_csv)
shutil.copy(output_csv, drive_path)
print(f"CSV 저장 완료: {drive_path}")

# 다운로드
from google.colab import files
files.download(output_csv)

## 7. context_narrative 형태로 변환

In [ ]:
# ⛔ 이 셀은 사용하지 않습니다.
# context_narrative 컬럼은 cell-18에서 생성되므로,
# DB INSERT는 반드시 cell-18 실행 후 cell-19에서 진행합니다.
print("cell-16 skip — cell-18 → cell-19 순서로 실행하세요.")

## 7. context_narrative 생성 → CSV 저장 → DB INSERT

**실행 순서:**
1. `cell-18` — situation + emotion + desire 합쳐서 `context_narrative` 컬럼 생성 + Drive 백업
2. `cell-19` — DB TRUNCATE 후 INSERT

In [ ]:
import shutil

# context_narrative 생성 (상황 + 감정 + 욕구 합치기)
df_results['context_narrative'] = (
    '상황: ' + df_results['situation'].fillna('') + ' ' +
    '감정: ' + df_results['emotion'].fillna('') + ' ' +
    '욕구: ' + df_results['desire'].fillna('')
)

# Drive 백업용 CSV 저장
db_csv = 'hyuck1_context_narrative.csv'
df_results[['context_num', 'start_sec', 'end_sec', 'context_narrative']].to_csv(
    db_csv, index=False, encoding='utf-8-sig'
)
shutil.copy(db_csv, os.path.join(BASE, db_csv))
print("context_narrative CSV 저장 완료!")

for _, row in df_results.head(3).iterrows():
    print(f"\n[씬 {row['context_num']}] {row['context_narrative'][:80]}...")


In [ ]:
import psycopg2
import uuid

# DB 연결
conn = psycopg2.connect(
    host="121.167.223.17",
    port=5432,
    dbname="hv02",
    user="postgres01",
    password="postgres01"
)
cur = conn.cursor()

# 기존 데이터 초기화
cur.execute("TRUNCATE TABLE public.analysis_scene_final;")
print("기존 데이터 초기화 완료")

# INSERT
job_id   = str(uuid.uuid4())
inserted = 0
skipped  = 0

for _, row in df_results.iterrows():
    if pd.isna(row.get('situation', '')) or str(row.get('situation', '')) == '':
        skipped += 1
        continue
    cur.execute(
        """
        INSERT INTO public.analysis_scene_final
            (job_id, scene_start_sec, scene_end_sec, context_narrative, created_at)
        VALUES (%s, %s, %s, %s, NOW())
        """,
        (job_id, float(row['start_sec']), float(row['end_sec']), row['context_narrative'])
    )
    inserted += 1

conn.commit()
cur.close()
conn.close()

print(f"DB INSERT 완료: {inserted}건 성공 / {skipped}건 스킵")
print(f"job_id: {job_id}")
